# Virtual mIF (GigaTIME-Flash) in TCGA-LUAD — full project notebook

**Author:** Almoutasm Bllah Alshawabkeh, MD · ORCID 0009-0005-7064-5239
**Registrations:** Study 1 pre-registered — [osf.io/kduvj](https://osf.io/kduvj) · Study 2 retrospectively registered — [osf.io/jdqas](https://osf.io/jdqas)

This notebook runs the whole project end to end:

| Part | What it does |
|---|---|
| 0 | Setup: installs, Google Drive, paths, constants |
| 1 | Helper functions (Gini, concentration, bootstrap CIs, partial correlation) |
| 2 | Cohort: slides, survival (clinical + follow-up), stage, exclusions, exploratory/confirmation split |
| 3 | GigaTIME-Flash: load model, mpp-aware tiling, pilot slide, checkpointed batch inference |
| 4 | Slide-level features for all 23 channels (mean, Gini, concentration) |
| 5 | Saltz TIL maps: slide-level analogues and within-slide multi-scale agreement (**Figure 2**) |
| 6 | Study 1 (pre-registered survival): gate, primary Cox, C-index, secondaries, forensics (**Table 2, Figure 4**) |
| 7 | Study 2 (molecular validation, per the 6 Sep 2026 plan): H1–H3, §5.3, §5.4 (**Table 3, Figure 3**) |
| 8 | Table 1 and Figure 1 (study design) |
| 9 | Export everything to Drive |

**How to use it.** Run top to bottom on Google Colab. Parts 3–5 need a GPU and many hours; every expensive step is checkpointed, so you can stop and resume. If you already have the per-tile feature files from the original run, set `RUN_INFERENCE = False` in Part 0 and the notebook skips straight to the analysis.

Every cell has a short note above it explaining what it does. Cells marked **⚠️ check** contain an assumption you should confirm against your original notebooks (currently only the Ayers gene list in 7.2).

## Part 0 — Setup

### 0.1 Install packages
Installs OpenSlide (reads `.svs` whole-slide images), lifelines (survival analysis), Hugging Face Hub (model download), and Google Cloud clients (BigQuery for the slide index, GCS for the slides). Run once per Colab session.

In [ ]:
!pip -q install openslide-python openslide-bin lifelines huggingface_hub google-cloud-bigquery google-cloud-storage gcsfs scikit-learn statsmodels openpyxl

### 0.2 Mount Google Drive and set paths
All inputs and outputs live in `MyDrive/luad_project/`. Sub-folders are created if they do not exist: `features/` (one CSV of per-tile values per slide), `til_maps/` (Saltz PNGs), `molecular/` (Thorsson, Xena, RPPA files) and `outputs/` (tables and figures).

In [ ]:
import os
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/luad_project')
except ImportError:                      # running locally
    ROOT = Path(os.environ.get('LUAD_ROOT', './luad_project'))

FEAT_DIR = ROOT / 'features'
TIL_DIR  = ROOT / 'til_maps_luad'        # continuous maps (TIL_maps_before_thres_v2)
MOL_DIR  = ROOT / 'molecular'
OUT_DIR  = ROOT / 'outputs'
for p in [FEAT_DIR, TIL_DIR, MOL_DIR, OUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)
print('Project root:', ROOT)

### 0.3 Constants and switches
Everything that was fixed in the registrations is set here once, so no later cell can quietly change it.

- `SEED = 42` — used for splits, bootstrap and cross-validation (as registered).
- Tile geometry: 128 µm of native tissue resized to 256 px; tissue threshold 10% (Study 1 registration).
- `GATE_RHO = 0.30` — the co-primary validation gate.
- Study 2 thresholds from the 6 Sep 2026 plan.
- `RUN_INFERENCE` — set to `False` if `features/` already holds the per-tile CSVs.
- `CHANNELS` — the 23 GigaTIME-Flash output channels in model order (from the model's `config.json`, dilution labels such as `_1:1000` removed). Part 3.1 re-reads and checks it.

In [ ]:
import numpy as np, pandas as pd, warnings, json, time, math
warnings.filterwarnings('ignore')
pd.set_option('display.width', 160)

SEED = 42
rng = np.random.default_rng(SEED)

# --- tiling (Study 1 registration, 'Variables') ---
TILE_UM         = 128      # microns of native tissue per tile
TILE_PX         = 256      # model input size
TISSUE_MIN_FRAC = 0.10     # keep tiles with >=10% tissue
SALTZ_PX_UM     = 50       # one Saltz-map pixel = 50 um patch

# --- registered decision rules ---
GATE_RHO   = 0.30                       # Study 1 co-primary gate
H1_RHO, H1_P = 0.25, 0.01               # Study 2 primary
H2_RHO, H2_P = 0.20, 0.05
H3_RHO, H3_P = 0.20, 0.05
N_BOOT = 2000

RUN_INFERENCE = False   # True = run GigaTIME-Flash on every slide (GPU, many hours)

# GigaTIME-Flash output order (index -> name); Part 3.1 re-reads it from the model's config.json
CHANNELS = ['DAPI', 'TRITC', 'Cy5', 'PD-1', 'CD14', 'CD4', 'T-bet', 'CD34', 'CD68', 'CD16', 'CD11c',
            'CD138', 'CD20', 'CD3', 'CD8', 'PD-L1', 'CK', 'Ki67', 'Tryptase', 'Actin-D',
            'Caspase3-D', 'PHH3-B', 'Transgelin']   # model order, from GigaTIME-Flash config.json
BATCH = 96              # L4 GPU; use 32 on a T4
print('Config loaded.')

## Part 1 — Helper functions

### 1.1 Statistics helpers
Small functions reused throughout:

- `gini(x)` — inequality of tile values within a slide (0 = even, 1 = concentrated). This is the registered dispersion feature.
- `conc(x)` — mean of the top-decile tiles divided by the slide mean (registered definition of `*_conc`).
- `zscore(s)` — standardise a column so hazard ratios are *per SD*.
- `spearman_boot(x, y)` — Spearman ρ, two-sided p, and a 2,000-iteration bootstrap 95% CI (seed 42).
- `partial_spearman(x, y, z)` — rank-based partial correlation of x and y given z (used in the shared-factor forensics).

In [ ]:
from scipy import stats

def gini(x):
    x = np.sort(np.asarray(x, float)); x = x[np.isfinite(x)]
    if len(x) == 0 or x.sum() == 0: return np.nan
    n = len(x); i = np.arange(1, n + 1)
    return (2 * np.sum(i * x) / (n * x.sum())) - (n + 1) / n

def conc(x):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    if len(x) == 0 or x.mean() == 0: return np.nan
    k = max(1, int(np.ceil(0.10 * len(x))))
    return np.sort(x)[-k:].mean() / x.mean()

def zscore(s):
    return (s - s.mean()) / s.std()

def spearman_boot(x, y, n_boot=N_BOOT, seed=SEED):
    d = pd.DataFrame({'x': x, 'y': y}).dropna()
    rho, p = stats.spearmanr(d.x, d.y)
    r = np.random.default_rng(seed); boots = []
    xv, yv = d.x.values, d.y.values
    for _ in range(n_boot):
        idx = r.integers(0, len(d), len(d))
        boots.append(stats.spearmanr(xv[idx], yv[idx])[0])
    lo, hi = np.nanpercentile(boots, [2.5, 97.5])
    return dict(rho=rho, lo=lo, hi=hi, p=p, n=len(d))

def partial_spearman(x, y, z):
    d = pd.DataFrame({'x': x, 'y': y, 'z': z}).dropna().rank()
    rx = d.x - np.polyval(np.polyfit(d.z, d.x, 1), d.z)
    ry = d.y - np.polyval(np.polyfit(d.z, d.y, 1), d.z)
    return stats.pearsonr(rx, ry)[0]

print('gini test (should be ~0.5):', round(gini(np.r_[np.zeros(50), np.ones(50)]), 3))

### 1.2 Plot style
One consistent look for every figure: white background, no top/right spines, 9-pt text, 300-dpi export. `savefig(name)` writes a PNG and a PDF to `outputs/`.

In [ ]:
import matplotlib, matplotlib.pyplot as plt
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'figure.dpi': 110})
BLUE, ORANGE, GREY, RED = '#1f5f99', '#d98c1f', '#9a9a9a', '#c0392b'

def savefig(name, fig=None):
    fig = fig or plt.gcf()
    for ext in ('png', 'pdf'):
        fig.savefig(OUT_DIR / f'{name}.{ext}', dpi=300, bbox_inches='tight')
    print('saved', name)

## Part 2 — Cohort

### 2.1 List TCGA-LUAD diagnostic slides (BigQuery)
Queries the ISB-CGC slide index for **diagnostic (DX)** H&E slides in TCGA-LUAD, with their GCS path and file size. You need a Google Cloud project ID for BigQuery billing (queries this small are free-tier). If you already have `cohort_full.csv`, this cell just loads it.

In [ ]:
GCP_PROJECT = 'your-gcp-project-id'   # ⚠️ set this
cohort_full_path = ROOT / 'cohort_full.csv'

if cohort_full_path.exists():
    slides = pd.read_csv(cohort_full_path)
    print('Loaded existing cohort_full.csv:', len(slides), 'rows')
else:
    from google.cloud import bigquery
    from google.colab import auth; auth.authenticate_user()
    bq = bigquery.Client(project=GCP_PROJECT)
    q = '''
    SELECT case_barcode AS patient_barcode, slide_barcode, file_name, file_size,
           file_gdc_id AS file_id, file_gcs_url
    FROM `isb-cgc-bq.TCGA.slide_images_gdc_current`
    WHERE project_short_name = 'TCGA-LUAD' AND slide_barcode LIKE '%-DX%'
    '''
    slides = bq.query(q).to_dataframe()
    slides['file_size_gb'] = slides.file_size / 1e9
    print('Diagnostic slides found:', len(slides))
slides.head()

### 2.2 Build overall survival from clinical **and** follow-up files
⚠️ Important: GDC `clinical.tsv` alone silently drops last-contact dates for about 310 living patients. Survival time must take the **latest** of `days_to_death`, `days_to_last_follow_up` (clinical) and `follow_ups.days_to_follow_up` (follow-up file). Event = 1 if vital status is Dead.

Download both TSVs from the GDC portal (TCGA-LUAD → Clinical → TSV) into the project folder first.

In [ ]:
def num(s):
    return pd.to_numeric(s.replace("'--", np.nan), errors='coerce')

clin = pd.read_csv(ROOT / 'clinical.tsv', sep='\t', low_memory=False)
fup  = pd.read_csv(ROOT / 'follow_up.tsv', sep='\t', low_memory=False)

pid = 'cases.submitter_id'
death = clin.groupby(pid)['demographic.days_to_death'].apply(lambda s: num(s).max())
lfu_c = clin.groupby(pid)['diagnoses.days_to_last_follow_up'].apply(lambda s: num(s).max())
lfu_f = fup.groupby(pid)['follow_ups.days_to_follow_up'].apply(lambda s: num(s).max())
vital = clin.groupby(pid)['demographic.vital_status'].first()

surv = pd.DataFrame({'death': death, 'lfu_c': lfu_c}).join(lfu_f.rename('lfu_f'), how='left')
surv['vital'] = vital
surv['event'] = (surv.vital == 'Dead').astype(int)
surv['time_days'] = np.where(surv.event == 1, surv.death, surv[['lfu_c', 'lfu_f']].max(axis=1))
surv = surv[surv.time_days > 0][['event', 'time_days']]
print('Patients with usable survival:', len(surv), '| deaths:', surv.event.sum())

### 2.3 Clinical covariates: age, sex and AJCC pathologic stage
Takes the first non-missing value per patient. Stage comes from the **primary** diagnosis row. `stage4` keeps I–IV; `stage_adv` is the registered binary split (I–II vs III–IV) used in the stage-adjusted model.

In [ ]:
def first_valid(s):
    s = s[~s.astype(str).isin(["'--", 'nan', 'Not Reported', 'Unknown'])]
    return s.iloc[0] if len(s) else np.nan

cov = clin.groupby(pid).agg(age=('demographic.age_at_index', first_valid),
                            sex=('demographic.sex_at_birth', first_valid))
prim = clin[clin['diagnoses.classification_of_tumor'].astype(str).str.contains('primary', case=False)]
stage_raw = prim.groupby(pid)['diagnoses.ajcc_pathologic_stage'].apply(first_valid)

def stage_group(s):
    s = str(s)
    for g in ['IV', 'III', 'II', 'I']:
        if f'Stage {g}' in s or s.strip() == g: return g
    return np.nan

cov['age'] = pd.to_numeric(cov.age, errors='coerce')
cov['stage4'] = stage_raw.map(stage_group)
cov['stage_adv'] = cov.stage4.map({'I': 0, 'II': 0, 'III': 1, 'IV': 1})
cov.stage4.value_counts(dropna=False)

### 2.4 Apply exclusions and keep one slide per patient
Joins slides to survival, keeps the first DX slide per patient, and flags slides with anomalously small files (≤0.01 GB — usually empty or corrupt). Slides without microns-per-pixel metadata fail at tiling and are dropped there. In the original run 9 of 465 eligible patients were excluded this way (one 0.01-GB slide was readable and kept), leaving 456. When the saved split is reloaded in 2.5, its exact membership is used.

In [ ]:
cohort = (slides.drop(columns=['event', 'time_days'], errors='ignore')
                .sort_values('slide_barcode')
                .drop_duplicates('patient_barcode')
                .merge(surv, left_on='patient_barcode', right_index=True, how='left'))
cohort_all = cohort.copy()            # before exclusions (used if the saved split is reloaded)
if 'file_size_gb' in cohort:
    small = cohort.file_size_gb <= 0.01
    print('Flagged as anomalously small (<=0.01 GB):', int(small.sum()))
    cohort = cohort[~small]
print('Eligible patients:', len(cohort))

### 2.5 Exploratory / confirmation split
The original project used a **58-patient exploratory set** (pipeline building, feature choice) and a **398-patient confirmation set** analysed once after registration. This cell reuses the saved split (`cohort_final.csv` = exploratory, `cohort_confirm.csv` = confirmation) so results reproduce exactly. If those files are absent it draws a new seeded split — but then results will not match the registration.

In [ ]:
ex_path, cf_path = ROOT / 'cohort_final.csv', ROOT / 'cohort_confirm.csv'
if ex_path.exists() and cf_path.exists():
    saved = pd.concat([pd.read_csv(ex_path), pd.read_csv(cf_path)])
    # the original exclusions were made by inspecting slides (missing mpp / unreadable files),
    # so membership is taken from the saved split rather than re-derived from file size
    cohort = cohort_all[cohort_all.patient_barcode.isin(saved.patient_barcode)].copy()
    ex_ids = set(pd.read_csv(ex_path).patient_barcode)
    cf_ids = set(pd.read_csv(cf_path).patient_barcode)
    # reuse the exact survival values used in the registered analysis
    sv = saved.drop_duplicates('patient_barcode').set_index('patient_barcode')[['event', 'time_days']]
    cohort = cohort.drop(columns=['event', 'time_days']).merge(sv, left_on='patient_barcode', right_index=True, how='left')
    rebuilt = surv.reindex(sv.index)
    print('Saved vs rebuilt survival agree for',
          int(((rebuilt.event == sv.event) & (np.isclose(rebuilt.time_days, sv.time_days))).sum()), 'of', len(sv), 'patients')
else:
    print('⚠️ No saved split found — drawing a new one (will not match the registration).')
    ids = cohort.patient_barcode.sample(frac=1, random_state=SEED).tolist()
    ex_ids, cf_ids = set(ids[:58]), set(ids[58:])

cohort['set'] = np.where(cohort.patient_barcode.isin(ex_ids), 'exploratory',
                 np.where(cohort.patient_barcode.isin(cf_ids), 'confirmation', 'unused'))
cohort = cohort[(cohort.set != 'unused') & cohort.event.notna()].merge(cov, left_on='patient_barcode', right_index=True, how='left')
print(cohort.groupby('set').agg(n=('patient_barcode', 'size'), deaths=('event', 'sum')))
assert not (ex_ids & cf_ids), 'exploratory and confirmation sets overlap!'

## Part 3 — GigaTIME-Flash inference

### 3.1 Load GigaTIME-Flash

**What this cell does:** downloads the GigaTIME-Flash weights from Hugging Face, builds the model from the authors' GitHub code, reads the channel order, loads every weight exactly, and runs a test tile. It prints one line per step (1–6) and, if something fails, says what went wrong and how to fix it.

**What the model does:** takes H&E tiles `[B, 3, 256, 256]` and returns 23 virtual-mIF maps `[B, 23, 256, 256]` (21 protein channels + 2 background). A sigmoid turns each output into a per-pixel probability.

**One-time setup before running:**
1. **Get access:** open huggingface.co/prov-gigatime/GigaTIME-flash and accept the terms.
2. **Store your token safely:** in Colab click the 🔑 icon → *Add new secret* → name it `HF_TOKEN`, paste a **read-only** token, and switch on *Notebook access*. Never paste a token into a cell.

**Two fixes built into this cell:**
- The model class (`GigaTIMEFlash`) is defined inside the authors' example notebook `scripts/gigatime_flash_testing.ipynb`, not in a `.py` file. The cell clones the repo and imports only the class and function definitions from that notebook.
- The checkpoint stores the patch-embedding layer as LoRA pieces (`base_layer`, `lora_A`, `lora_B`), while the freshly built model expects one plain layer. Without a fix, that first layer silently keeps its starting values. The cell merges the pieces (W = W_base + scale × B·A) and then loads with a **strict** check, which fails loudly if any weight still mismatches.

**What you should see:** `Merged LoRA into 1 layer(s) (scale 2.0)`, `✓ All weights loaded exactly (strict check passed)`, and `✓ Output shape (1, 23, 256, 256)`. Then confirm on real tissue with the pilot in 3.4.

In [ ]:
# === 3.1 Load GigaTIME-Flash: prints every step and explains any error ===
import os, sys, json, glob, re, ast, subprocess, importlib.util
import torch
from huggingface_hub import login, snapshot_download

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
REPO_ID  = 'prov-gigatime/GigaTIME-flash'                     # weights (Hugging Face)
CODE_URL = 'https://github.com/prov-gigatime/gigatime.git'    # model code (GitHub)
CODE_DIR = '/content/gigatime'
model, local_dir = None, None


def step(n, msg):
    print(f'[{n}/6] {msg}')


def fail(msg, err=None):
    print(f'   ❌ {msg}')
    if err is not None:
        print(f'   Error: {type(err).__name__}: {err}')
    print('   → Stopped here. Fix the problem above and re-run this cell.')


def build_model():
    # The weights file has no architecture: get the authors' code and find GigaTIMEFlash
    if not os.path.exists(CODE_DIR):
        print('   Cloning GigaTIME repo…')
        subprocess.run(['git', 'clone', '-q', CODE_URL, CODE_DIR], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'peft'], check=False)
    sys.path.insert(0, CODE_DIR)
    pattern = re.compile(r'class\s+(GigaTIME_?Flash\w*)\s*[\(:]', re.IGNORECASE)

    # A) a .py file that defines the class
    for f in glob.glob(f'{CODE_DIR}/**/*.py', recursive=True):
        m = pattern.search(open(f, errors='ignore').read())
        if m:
            cls = m.group(1)
            print(f'   Found class {cls} in {f.replace(CODE_DIR + "/", "")}')
            sys.path.insert(0, os.path.dirname(f))
            spec = importlib.util.spec_from_file_location('gt_flash_mod', f)
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            return getattr(mod, cls)(num_classes=23)

    # B) the authors' notebook: run only its imports, functions and classes
    for nb_path in glob.glob(f'{CODE_DIR}/**/*.ipynb', recursive=True):
        cells = [''.join(c['source']) for c in json.load(open(nb_path))['cells']
                 if c['cell_type'] == 'code']
        if not any(pattern.search(c) for c in cells):
            continue
        print(f'   Found the class inside notebook {nb_path.replace(CODE_DIR + "/", "")}')
        ns = {}
        sys.path.insert(0, os.path.dirname(nb_path))
        for src in cells:
            src = '\n'.join(l for l in src.splitlines() if not l.strip().startswith(('!', '%')))
            try:
                tree = ast.parse(src)
            except SyntaxError:
                continue
            for node in tree.body:
                if isinstance(node, (ast.Import, ast.ImportFrom, ast.ClassDef, ast.FunctionDef)):
                    try:
                        exec(compile(ast.Module(body=[node], type_ignores=[]), nb_path, 'exec'), ns)
                    except Exception as e:
                        print(f'   ⚠️ skipped one definition ({type(e).__name__}: {e})')
            m = pattern.search(src)
            if m:
                cls = m.group(1)
                print(f'   Using class {cls}')
                return ns[cls](num_classes=23)

    raise ImportError('Could not find the GigaTIME-Flash class in the repo.')


def merge_lora_keys(state, model):
    # The checkpoint stores some layers as LoRA (base_layer + lora_A + lora_B) while the
    # built model expects a plain layer. Merge them: W = W_base + scale * (B @ A).
    model_keys = set(model.state_dict().keys())
    scale = next((mod.scaling['default'] for mod in model.modules()
                  if isinstance(getattr(mod, 'scaling', None), dict) and 'default' in mod.scaling), None)
    merged = []
    for k in [k for k in list(state) if k.endswith('.base_layer.weight')]:
        pre = k[:-len('base_layer.weight')]
        if pre + 'weight' not in model_keys:
            continue                                  # model also uses LoRA here: leave as is
        if scale is None:
            raise RuntimeError('Could not find the LoRA scaling factor in the model.')
        W = state.pop(pre + 'base_layer.weight')
        A = state.pop(pre + 'lora_A.default.weight')
        B = state.pop(pre + 'lora_B.default.weight')
        state[pre + 'weight'] = W + (B.flatten(1) @ A.flatten(1)).view_as(W) * scale
        if pre + 'base_layer.bias' in state:
            state[pre + 'bias'] = state.pop(pre + 'base_layer.bias')
        merged.append(pre.rstrip('.'))
    return state, merged, scale


def run():
    global model, local_dir, CHANNELS

    # 1. Device
    step(1, f'Device: {DEVICE}')
    if DEVICE == 'cpu':
        print('   ⚠️ No GPU. Runtime → Change runtime type → GPU.')

    # 2. Token from Colab Secrets (never hard-code it)
    step(2, 'Reading HF_TOKEN from Colab Secrets…')
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    except Exception as e:
        token = os.environ.get('HF_TOKEN')
        if not token:
            return fail('No HF_TOKEN. In Colab: 🔑 icon → Add new secret → name HF_TOKEN → '
                        'paste a READ-ONLY token → switch on "Notebook access".', e)
    if not token:
        return fail('HF_TOKEN is empty.')
    os.environ['HF_TOKEN'] = token
    print('   ✓ Token found (not shown).')

    # 3. Log in and download the weights
    step(3, f'Downloading {REPO_ID}…')
    try:
        login(token=token, add_to_git_credential=False)
        local_dir = snapshot_download(repo_id=REPO_ID)
        print(f'   ✓ Downloaded. Files: {sorted(os.listdir(local_dir))}')
    except Exception as e:
        if any(k in str(e).lower() for k in ['401', '403', 'gated']):
            return fail('Access denied. Accept the terms at huggingface.co/prov-gigatime/GigaTIME-flash.', e)
        return fail('Download failed.', e)

    # 4. Channel order from config.json ('CD3_1:1000' -> 'CD3')
    step(4, 'Reading channel order from config.json…')
    cfg = json.load(open(os.path.join(local_dir, 'config.json')))
    names = next((cfg[k] for k in ['channels', 'channel_names', 'output_channels',
                                   'markers', 'mif_channels'] if k in cfg), None)
    if names is None or len(names) != 23:
        print('   ⚠️ Could not read 23 channel names. config.json keys:', list(cfg.keys()))
    else:
        CHANNELS = [str(n).split('_1:')[0] for n in names]
        print('   ✓ 23 channels:', ', '.join(f'{i}={n}' for i, n in enumerate(CHANNELS)))
    expected = {8: 'CD68', 13: 'CD3', 14: 'CD8', 15: 'PD-L1', 16: 'CK'}
    bad = [i for i, n in expected.items() if CHANNELS[i] != n]
    print('   ✓ Matches original pipeline (8 CD68, 13 CD3, 14 CD8, 15 PD-L1, 16 CK)' if not bad
          else f'   ⚠️ Channel positions differ from the original pipeline at {bad}')

    # 5. Build the model, merge LoRA layers, load weights with a strict check
    step(5, 'Building model and loading weights…')
    try:
        m = build_model()
    except Exception as e:
        return fail('Building the model failed. Paste this error to Claude.', e)
    try:
        state = torch.load(os.path.join(local_dir, 'model.pth'), map_location='cpu')
        if isinstance(state, dict) and 'state_dict' in state:
            state = state['state_dict']
        state, merged, scale = merge_lora_keys(dict(state), m)
        if merged:
            print(f'   Merged LoRA into {len(merged)} layer(s) (scale {scale}): {merged}')
        m.load_state_dict(state, strict=True)        # fails loudly if any key still mismatches
        model = m.to(DEVICE).eval()
        print('   ✓ All weights loaded exactly (strict check passed).')
    except Exception as e:
        return fail('Loading weights failed: some layers still do not match. Paste this error to Claude.', e)

    # 6. Smoke test with one blank tile
    step(6, 'Running a test tile through the model…')
    try:
        with torch.no_grad():
            out = model(torch.zeros(1, 3, 256, 256, device=DEVICE))
        if isinstance(out, (tuple, list)):
            out = out[0]
        shape = tuple(out.shape)
        if shape == (1, 23, 256, 256):
            print(f'   ✓ Output shape {shape}. GigaTIME-Flash is ready.')
        else:
            print(f'   ⚠️ Unexpected output shape {shape} (expected (1, 23, 256, 256)).')
    except Exception as e:
        return fail('The test forward pass failed.', e)


run()

### 3.2 Magnification-aware tiling
Reads the slide's microns-per-pixel (mpp) and cuts **128 µm** tiles at native resolution, then resizes each to 256 px. ⚠️ This matters: TCGA-LUAD slides are mostly 40× (mpp ≈ 0.25), and reading fixed 256-px tiles without checking mpp feeds the model a field at twice the intended magnification — a bug in the authors' own TCGA demo.

Tissue detection: a tile is kept if ≥10% of its pixels have saturation > 0.07 on a thumbnail-scale check. Images are ImageNet-normalised.

In [ ]:
import openslide
from PIL import Image

IMNET_MEAN = np.array([0.485, 0.456, 0.406]); IMNET_STD = np.array([0.229, 0.224, 0.225])

def slide_mpp(sl):
    mpp = sl.properties.get(openslide.PROPERTY_NAME_MPP_X)
    return float(mpp) if mpp else None

def tissue_fraction(rgb):
    hsv = np.asarray(Image.fromarray(rgb).convert('HSV'), float) / 255
    return ((hsv[..., 1] > 0.07) & (hsv[..., 2] < 0.95)).mean()

def iter_tiles(sl):
    mpp = slide_mpp(sl)
    if mpp is None:
        raise ValueError('no mpp metadata — exclude slide')
    size0 = int(round(TILE_UM / mpp))           # tile edge in level-0 pixels
    W, H = sl.dimensions
    for y in range(0, H - size0 + 1, size0):
        for x in range(0, W - size0 + 1, size0):
            rgb = np.asarray(sl.read_region((x, y), 0, (size0, size0)).convert('RGB'))
            small = np.asarray(Image.fromarray(rgb).resize((64, 64)))
            tf = tissue_fraction(small)
            if tf < TISSUE_MIN_FRAC:
                continue
            img = np.asarray(Image.fromarray(rgb).resize((TILE_PX, TILE_PX), Image.BILINEAR)) / 255.
            yield x, y, tf, ((img - IMNET_MEAN) / IMNET_STD).transpose(2, 0, 1).astype(np.float32)

### 3.3 Run the model on one slide
Batches tiles, runs the model, applies sigmoid, and stores the **mean probability per channel per tile** with the tile's level-0 position. Output: one CSV per slide in `features/` (columns `x, y, tissue_frac, <23 channels>`, plus `mpp`).

In [ ]:
@torch.no_grad()
def infer_slide(svs_path, model, batch=BATCH):
    sl = openslide.OpenSlide(str(svs_path)); mpp = slide_mpp(sl)
    rows, buf, meta = [], [], []
    def flush():
        x = torch.from_numpy(np.stack(buf)).to(DEVICE)
        with torch.autocast(DEVICE, enabled=(DEVICE == 'cuda')):
            out = model(x)
            out = out[0] if isinstance(out, (tuple, list)) else out
            prob = torch.sigmoid(out.float())               # (B, 23, H, W)
        means = prob.mean(dim=(2, 3)).cpu().numpy()
        for (tx, ty, tf), m in zip(meta, means):
            rows.append([tx, ty, tf, *m])
        buf.clear(); meta.clear()
    for tx, ty, tf, arr in iter_tiles(sl):
        buf.append(arr); meta.append((tx, ty, tf))
        if len(buf) == batch: flush()
    if buf: flush()
    df = pd.DataFrame(rows, columns=['x', 'y', 'tissue_frac', *CHANNELS]); df['mpp'] = mpp
    return df

### 3.4 Pilot: one slide end to end, with a picture
Runs the full pipeline on the first confirmation slide and draws the H&E thumbnail next to the virtual CD8 map. Use this to sanity-check tiling and orientation before the long batch run. **Check the model is reading tissue:** CK should be bright over tumour epithelium and dark in stroma; CD8 should appear as scattered foci, denser at the invasive margin and in lymphoid aggregates. Noise or uniform blur means the weights did not load correctly.

In [ ]:
def download_slide(row, dest='/content/tmp_slide.svs'):
    import gcsfs
    fs = gcsfs.GCSFileSystem(token='anon')
    url = row.get('file_gcs_url') or f"gs://gdc-tcga-phs000178-open/{row['file_id']}/{row['file_name']}"
    fs.get(url, dest); return dest

def plot_virtual_map(df, channel='CD8', svs_path=None, ax=None):
    ax = ax or plt.gca()
    step = df.x.diff().abs().replace(0, np.nan).min()
    gx, gy = (df.x / step).astype(int), (df.y / step).astype(int)
    grid = np.full((gy.max() + 1, gx.max() + 1), np.nan); grid[gy, gx] = df[channel]
    im = ax.imshow(grid, cmap='magma'); ax.set_title(f'Virtual {channel}'); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.04)

if RUN_INFERENCE:
    row = cohort[cohort.set == 'confirmation'].iloc[0]
    p = download_slide(row); t0 = time.time()
    pilot = infer_slide(p, model)
    print(f"{row.slide_barcode}: {len(pilot)} tiles in {time.time()-t0:.0f}s")
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    axes[0].imshow(openslide.OpenSlide(p).get_thumbnail((600, 600))); axes[0].axis('off'); axes[0].set_title('H&E')
    plot_virtual_map(pilot, 'CD8', ax=axes[1]); plot_virtual_map(pilot, 'CK', ax=axes[2])
    savefig('pilot_virtual_CD8_CK')

### 3.5 Batch inference with checkpointing
Streams each slide: download → infer → save per-tile CSV → delete the slide (Colab disk is small). Slides already in `features/` are skipped, so you can restart after a disconnect. Failures are retried up to 3 times with back-off and logged to `failures.csv`.

In [ ]:
def run_batch(df_cohort):
    fails = []
    todo = [r for _, r in df_cohort.iterrows() if not (FEAT_DIR / f'{r.slide_barcode}.csv').exists()]
    print(f'{len(todo)} slides to process')
    for i, r in enumerate(todo, 1):
        for attempt in range(3):
            try:
                p = download_slide(r)
                df = infer_slide(p, model); df.to_csv(FEAT_DIR / f'{r.slide_barcode}.csv', index=False)
                os.remove(p); print(f'[{i}/{len(todo)}] {r.slide_barcode} ok ({len(df)} tiles)')
                break
            except Exception as e:
                wait = 30 * (attempt + 1); print(f'  retry {attempt+1} {r.slide_barcode}: {e}'); time.sleep(wait)
        else:
            fails.append({'slide_barcode': r.slide_barcode, 'error': str(e)})
    if fails: pd.DataFrame(fails).to_csv(ROOT / 'failures.csv', index=False)

if RUN_INFERENCE:
    run_batch(cohort)

## Part 4 — Slide-level features

### 4.1 Aggregate tiles into slide features (all 23 channels)
For every slide and channel, computes the **mean** (Study 2 features), **Gini** and **concentration** (Study 1 features) over tissue tiles. Result: `slide_features_all.csv`, one row per patient. ⚠️ If your original per-tile files used different column names, rename them to the `CHANNELS` names first.

In [ ]:
# === 4.1 Aggregate tiles into slide features (all 23 channels) ===
def load_tiles(path):
    """Read one per-tile CSV and standardise it: 'mean_CD8' -> 'CD8'; drop the file's own grid columns."""
    t = pd.read_csv(path)
    t = t.rename(columns={c: c[len('mean_'):] for c in t.columns if c.startswith('mean_')})
    return t.drop(columns=[c for c in ['gx', 'gy'] if c in t])

def aggregate_slide(path):
    t = load_tiles(path)
    out = {'slide_barcode': path.stem, 'n_tiles': len(t)}
    for c in CHANNELS:
        if c in t:
            v = t[c].values
            out[f'{c}_mean'], out[f'{c}_gini'], out[f'{c}_conc'] = v.mean(), gini(v), conc(v)
    return out

feat_files = sorted(FEAT_DIR.glob('*.csv'))
if feat_files:
    first = load_tiles(feat_files[0])
    found = [c for c in CHANNELS if c in first]
    print(f'{len(feat_files)} per-tile files | channels recognised: {len(found)}/23')
    if len(found) < 23:
        print('⚠️ Missing channels:', [c for c in CHANNELS if c not in first])
    feats = pd.DataFrame([aggregate_slide(p) for p in feat_files])
    feats['patient_barcode'] = feats.slide_barcode.str[:12]
    feats.to_csv(ROOT / 'slide_features_all.csv', index=False)
    print('Slide features:', feats.shape)
else:
    print('No per-tile files in features/. Falling back to saved slide-level files below.')

### 4.2 Build the analysis dataset (and reuse the registered Saltz values)
Joins the slide features to the cohort. If no per-tile files are available, it loads the slide-level files saved by the original run instead: `confirm_features.csv` (confirmation set) and `normalized_features.csv` (exploratory set).

It also takes the **Saltz slide-level analogues** (`saltz_gini`, etc.) from those saved files, so the co-primary validation gate reproduces the registered value (ρ = 0.146). Recomputing them from the Saltz map images in 5.2 selects slightly different pixels and gives ρ ≈ 0.22; both fail the 0.30 gate, but the paper reports the registered pipeline. Cell 5.2 only recomputes them if no saved values exist.

In [ ]:
# === 4.2 Load saved slide-level files and build the analysis dataset ===
cf_path, ex_path = ROOT / 'confirm_features.csv', ROOT / 'normalized_features.csv'
if not feat_files:
    cf = pd.read_csv(cf_path)
    ex = pd.read_csv(ex_path)
    feats = pd.concat([cf, ex], ignore_index=True).drop(columns=['event', 'time_days'], errors='ignore')
    print('Loaded saved features:', feats.shape)

# Reuse the Saltz slide-level analogues computed by the ORIGINAL registered pipeline,
# so the co-primary gate reproduces the registered value (rho = 0.146). Recomputing them
# from the Saltz PNGs (cell 5.2) gives a slightly different pixel selection.
saved_saltz = []
for p in [cf_path, ex_path]:
    if p.exists():
        s = pd.read_csv(p)
        cols = [c for c in s.columns if c.startswith('saltz_')]
        if cols:
            saved_saltz.append(s[['patient_barcode'] + cols])
if saved_saltz:
    saved_saltz = pd.concat(saved_saltz).drop_duplicates('patient_barcode')
    feats = feats.drop(columns=[c for c in feats.columns if c.startswith('saltz_')], errors='ignore') \
                 .merge(saved_saltz, on='patient_barcode', how='left')
    print('Using saved Saltz analogues from the original pipeline:', [c for c in saved_saltz.columns if c != 'patient_barcode'],
          '| patients:', saved_saltz.patient_barcode.nunique())
else:
    print('No saved Saltz analogues found; cell 5.2 will recompute them from the Saltz maps.')

data = cohort.merge(feats.drop(columns=['slide_barcode'], errors='ignore'), on='patient_barcode', how='inner')
CONF = data[data.set == 'confirmation'].copy(); EXPL = data[data.set == 'exploratory'].copy()
print('Confirmation:', len(CONF), 'patients,', CONF.event.sum(), 'deaths | Exploratory:', len(EXPL))

## Part 5 — Saltz TIL maps (independent reference)

### 5.1 Read a Saltz map
Saltz et al. (Cell Reports 2018) TIL maps from TCIA (DOI 10.7937/K9/TCIA.2018.Y75F9W1), version `TIL_maps_before_thres_v2`. Each is an RGB PNG: **red = continuous TIL probability (0–255)**, **blue = tissue mask**, green ignored. One pixel = one 50-µm patch.

In [ ]:
def read_saltz(slide_barcode):
    f = TIL_DIR / f'{slide_barcode}.png'
    if not f.exists(): return None
    a = np.asarray(Image.open(f).convert('RGB')).astype(float)
    til = a[..., 0] / 255.; tissue = a[..., 2] > 0
    return np.where(tissue, til, np.nan)

### 5.2 Slide-level Saltz analogues
For each slide, the same statistics as the virtual features (Gini, concentration, mean) on TIL-probability pixels inside tissue. `saltz_gini` is the reference for the registered gate. Skipped if the saved features already contain these columns.

In [ ]:
if 'saltz_gini' not in data:
    rows = []
    for sb in data.slide_barcode:
        m = read_saltz(sb)
        if m is None: continue
        v = m[np.isfinite(m)]
        rows.append({'slide_barcode': sb, 'saltz_gini': gini(v), 'saltz_conc': conc(v), 'saltz_mean': v.mean()})
    data = data.merge(pd.DataFrame(rows), on='slide_barcode', how='left')
    CONF = data[data.set == 'confirmation'].copy(); EXPL = data[data.set == 'exploratory'].copy()
print('Slides with Saltz analogues:', data.saltz_gini.notna().sum())

### 5.3 Within-slide agreement at increasing scale (Figure 2)
Maps every virtual tile and every Saltz pixel onto a common grid (128 µm → 1 mm cells), averages each grid cell, and computes the Spearman ρ between virtual marker and Saltz TIL probability **within each slide**. The median across slides is plotted per scale. In the original run ρ rose from ≈0.25 at tile level to ≈0.56 at 1 mm, with CD3 > CD8 > CD68 and CK as a clean negative control. Needs per-tile files.

In [ ]:
# === 5.3 Within-slide agreement with Saltz maps at increasing scale (Figure 2) ===
SCALES_UM = [128, 256, 512, 1024]
MARKERS   = ['CD3', 'CD8', 'CD68', 'CK']

# --- 1. microns-per-pixel for each slide (original per-tile files don't store it) ---
mpp_lookup = {}
for f in ['slide_info.csv', 'slide_info_confirm.csv']:
    if (ROOT / f).exists():
        si = pd.read_csv(ROOT / f)
        id_col  = next((c for c in si.columns if 'slide' in c.lower() or 'barcode' in c.lower()), si.columns[0])
        mpp_col = next((c for c in si.columns if 'mpp' in c.lower()), None)
        if mpp_col:
            mpp_lookup.update(dict(zip(si[id_col].astype(str), si[mpp_col])))
print(f'mpp values found for {len(mpp_lookup)} slides')

def get_mpp(tile_df, slide_id):
    if 'mpp' in tile_df:
        return float(tile_df['mpp'].iloc[0])
    for key, val in mpp_lookup.items():
        if key == slide_id or slide_id.startswith(key) or key.startswith(slide_id[:12]):
            return float(val)
    return None

# --- 2. per-slide, per-scale agreement ---
def within_slide_rho(tile_df, saltz, mpp, scale_um, marker):
    gx = (tile_df['x'] * mpp // scale_um).astype(int)
    gy = (tile_df['y'] * mpp // scale_um).astype(int)
    v = tile_df.assign(gx=gx, gy=gy).groupby(['gy', 'gx'])[marker].mean()
    sy, sx = np.nonzero(np.isfinite(saltz))
    s = pd.DataFrame({'gy': (sy * SALTZ_PX_UM // scale_um).astype(int),
                      'gx': (sx * SALTZ_PX_UM // scale_um).astype(int),
                      'til': saltz[sy, sx]}).groupby(['gy', 'gx']).til.mean()
    j = pd.concat([v, s], axis=1, join='inner').dropna()
    return stats.spearmanr(j.iloc[:, 0], j.iloc[:, 1])[0] if len(j) > 20 else np.nan

res, skipped = [], {'no Saltz map': 0, 'no mpp': 0, 'marker missing': 0}
checked_units = False
for p in feat_files:
    sal = read_saltz(p.stem)
    if sal is None:
        skipped['no Saltz map'] += 1; continue
    t = load_tiles(p)                                   # 'mean_CD8' -> 'CD8'
    mpp = get_mpp(t, p.stem)
    if mpp is None:
        skipped['no mpp'] += 1; continue
    if not checked_units:                               # x/y must be level-0 pixels
        step_um = np.median(np.diff(np.sort(t['x'].unique()))) * mpp
        print(f'Tile spacing check: {step_um:.0f} µm between neighbouring tiles (expected ≈128)')
        if not 100 < step_um < 160:
            print('⚠️ x/y do not look like level-0 pixels: the scale mapping below may be wrong. Tell Claude.')
        checked_units = True
    for mk in MARKERS:
        if mk not in t:
            skipped['marker missing'] += 1; continue
        for sc in SCALES_UM:
            res.append({'slide': p.stem, 'scale': sc, 'marker': mk,
                        'rho': within_slide_rho(t, sal, mpp, sc, mk)})

print('Slides used:', len({r['slide'] for r in res}), '| skipped:', skipped)

# --- 3. Figure 2 ---
if res:
    ws = pd.DataFrame(res); ws.to_csv(OUT_DIR / 'within_slide_agreement.csv', index=False)
    summ = ws.groupby(['marker', 'scale']).rho.median().unstack(0)
    print(summ.round(3))
    fig, ax = plt.subplots(figsize=(4.5, 3.4))
    for mk, col in zip(MARKERS, [BLUE, ORANGE, GREY, RED]):
        if mk in summ:
            ax.plot(summ.index, summ[mk], 'o-', color=col, label=mk)
    ax.axhline(0, color='k', lw=0.6); ax.set_xscale('log', base=2)
    ax.set_xticks(SCALES_UM); ax.set_xticklabels(['128 µm', '256 µm', '512 µm', '1 mm'])
    ax.set_ylabel('Median within-slide Spearman ρ\nvs Saltz TIL probability'); ax.legend(frameon=False)
    ax.set_title('Figure 2. Within-slide agreement'); savefig('Figure2_within_slide'); plt.show()
else:
    print('❌ No slides could be analysed. Paste the output above to Claude.')

## Part 6 — Study 1: pre-registered survival analysis (osf.io/kduvj)

Everything below follows the registration exactly: primary feature **CD8_gini**, univariable Cox per SD, success = HR < 1 with p < 0.05, co-primary gate ρ ≥ 0.30 vs `saltz_gini`. All analyses use the **confirmation set only**, except the flagged pooled analysis.

### 6.1 Co-primary validation gate
Spearman ρ between virtual `CD8_gini` and `saltz_gini` across confirmation slides. Registered rule: if ρ < 0.30, the survival result is reported as **unresolved**, whatever its p-value.

In [ ]:
gate = spearman_boot(CONF.CD8_gini, CONF.saltz_gini)
gate_pass = gate['rho'] >= GATE_RHO
print(f"Gate: rho = {gate['rho']:.3f} (95% CI {gate['lo']:.3f} to {gate['hi']:.3f}), n = {gate['n']}  ->  {'PASS' if gate_pass else 'FAIL (survival result = unresolved)'}")

### 6.2 Primary analysis: univariable Cox on standardised CD8_gini
Hazard ratio per SD of CD8_gini, 95% CI and p. Expected from the original run: HR 0.997 (0.84–1.18), p = 0.97.

In [ ]:
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test, proportional_hazard_test
from lifelines.utils import concordance_index

def cox_per_sd(df, feature, extra=None, strata=None):
    cols = [feature] + (extra or []) + (strata or []) + ['time_days', 'event']
    d = df[cols].dropna().copy(); d[feature] = zscore(d[feature])
    m = CoxPHFitter().fit(d, 'time_days', 'event', strata=strata)
    s = m.summary.loc[feature]
    return dict(feature=feature, HR=s['exp(coef)'], lo=s['exp(coef) lower 95%'],
                hi=s['exp(coef) upper 95%'], p=s['p'], n=len(d), events=int(d.event.sum())), m, d

prim, prim_model, prim_d = cox_per_sd(CONF, 'CD8_gini')
print(f"Primary: HR {prim['HR']:.3f} ({prim['lo']:.2f}-{prim['hi']:.2f}), p = {prim['p']:.2f}, n = {prim['n']}, events = {prim['events']}")
print('Registered success (HR<1 and p<0.05):', prim['HR'] < 1 and prim['p'] < 0.05)

### 6.3 Proportional-hazards check
Schoenfeld-residual test for the primary model (registered). If p < 0.05 the registration calls for a time-split sensitivity model alongside; the primary conclusion still rests on the pre-specified model.

In [ ]:
ph = proportional_hazard_test(prim_model, prim_d, time_transform='rank')
print(ph.summary[['test_statistic', 'p']])

### 6.4 Out-of-fold C-index (5-fold, seed 42)
Fits the Cox model on 4/5 of the confirmation set and scores the held-out fifth, five times; the concordance of all out-of-fold risk scores is reported. 0.5 = chance. ⚠️ This is the registered recipe (plain `KFold`, shuffled, seed 42); a stratified split is printed for comparison because the original notebook reported 0.465 and this recipe gives ≈0.425.

In [ ]:
from sklearn.model_selection import KFold, StratifiedKFold

def oof_cindex(df, feature, splitter):
    d = df[[feature, 'time_days', 'event']].dropna().reset_index(drop=True); risk = np.zeros(len(d))
    for tr, te in splitter(d):
        m = CoxPHFitter().fit(d.iloc[tr], 'time_days', 'event')
        risk[te] = m.predict_partial_hazard(d.iloc[te]).values
    return concordance_index(d.time_days, -risk, d.event)   # negate: higher hazard = shorter survival

c_kf  = oof_cindex(CONF, 'CD8_gini', lambda d: KFold(5, shuffle=True, random_state=SEED).split(d))
c_skf = oof_cindex(CONF, 'CD8_gini', lambda d: StratifiedKFold(5, shuffle=True, random_state=SEED).split(d, d.event))
print(f'OOF C-index, KFold (registered): {c_kf:.3f} | StratifiedKFold: {c_skf:.3f}')

### 6.5 Pre-specified secondary analyses
All descriptive, no success criteria (registration §5.3):

- **(a)** CD8_conc, PD-L1_gini, CD68_gini — univariable Cox + gate correlation with the matching Saltz statistic.
- **(b)** Stage-adjusted Cox for CD8_gini, stage **I–II vs III–IV**.
- **(c)** Kaplan–Meier, a priori median split, log-rank test.
- **(d)** Pooled exploratory + confirmation, stratified by set (flagged: partly in-sample). The registration said 465; 456 were analysable after exclusions.
- **(e)** Negative control CK_gini through the identical pipeline (expected HR ≈ 1).

In [ ]:
rows = [dict(analysis='Primary', **prim)]
for f in ['CD8_conc', 'PD-L1_gini', 'CD68_gini']:
    r, _, _ = cox_per_sd(CONF, f); rows.append(dict(analysis='Secondary', **r))
r, _, _ = cox_per_sd(CONF, 'CK_gini'); rows.append(dict(analysis='Negative control', **r))
r, _, _ = cox_per_sd(CONF, 'CD8_gini', extra=['stage_adv'])
rows.append(dict(analysis='Stage-adjusted (I-II vs III-IV)', **r))

pooled = pd.concat([EXPL.assign(z=zscore(EXPL.CD8_gini), s=0), CONF.assign(z=zscore(CONF.CD8_gini), s=1)])
mp = CoxPHFitter().fit(pooled[['z', 's', 'time_days', 'event']].dropna(), 'time_days', 'event', strata=['s'])
sp = mp.summary.loc['z']
rows.append(dict(analysis='Pooled, stratified by set', feature='CD8_gini', HR=sp['exp(coef)'],
                 lo=sp['exp(coef) lower 95%'], hi=sp['exp(coef) upper 95%'], p=sp['p'],
                 n=len(pooled), events=int(pooled.event.sum())))

hi_grp = CONF.CD8_gini > CONF.CD8_gini.median()
lr = logrank_test(CONF.time_days[hi_grp], CONF.time_days[~hi_grp], CONF.event[hi_grp], CONF.event[~hi_grp])
km_meds = {g: KaplanMeierFitter().fit(CONF.time_days[m] / 30.44, CONF.event[m]).median_survival_time_
           for g, m in [('above', hi_grp), ('at/below', ~hi_grp)]}

table2 = pd.DataFrame(rows)
print(table2.round(3).to_string(index=False))
print(f"\nKM median split: log-rank p = {lr.p_value:.2f}; median OS {km_meds['above']:.1f} vs {km_meds['at/below']:.1f} months")

### 6.6 Gate correlations for every Study 1 feature
Spearman ρ of each virtual dispersion feature with its Saltz analogue (registered under secondary analysis (a)). Only CD8_gini is the formal gate.

In [ ]:
gates = []
for f, s in [('CD8_gini', 'saltz_gini'), ('CD8_conc', 'saltz_conc'), ('CD3_gini', 'saltz_gini'),
             ('PD-L1_gini', 'saltz_gini'), ('CD68_gini', 'saltz_gini'), ('CK_gini', 'saltz_gini')]:
    if f in CONF and s in CONF:
        g = spearman_boot(CONF[f], CONF[s]); gates.append({'feature': f, 'vs': s, **g})
pd.DataFrame(gates).round(3)

### 6.7 Exploratory-set estimate (for the winner's-curse comparison)
The same primary model in the 58-patient exploratory set. Original run: HR 0.71, p = 0.12, gate ρ = 0.40 — a signal that did not replicate in the powered confirmation set.

In [ ]:
ex_res, _, _ = cox_per_sd(EXPL, 'CD8_gini')
ex_gate = stats.spearmanr(EXPL.CD8_gini, EXPL.saltz_gini)[0]
print(f"Exploratory: HR {ex_res['HR']:.2f} ({ex_res['lo']:.2f}-{ex_res['hi']:.2f}), p = {ex_res['p']:.2f}, gate rho = {ex_gate:.2f}, n = {ex_res['n']}")

### 6.8 Shared-factor forensics (registered §5.3f)
Checks whether dispersion features move together regardless of biology — the artefact that made slide **means** look prognostic. Prints the inter-feature Spearman matrix and draws it as a heatmap (**Supplementary Figure S1**). High correlations of immune features with CK (a structural control) indicate shared tissue-patchiness variance.

In [ ]:
disp = [c for c in ['CD8_gini', 'CD3_gini', 'CD68_gini', 'PD-L1_gini', 'CK_gini', 'saltz_gini'] if c in CONF]
cm = CONF[disp].corr(method='spearman')
fig, ax = plt.subplots(figsize=(4.8, 4))
im = ax.imshow(cm, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(disp))); ax.set_xticklabels(disp, rotation=45, ha='right'); ax.set_yticks(range(len(disp))); ax.set_yticklabels(disp)
for i in range(len(disp)):
    for j in range(len(disp)):
        ax.text(j, i, f'{cm.iloc[i, j]:.2f}', ha='center', va='center', fontsize=7, color='white' if abs(cm.iloc[i, j]) > 0.6 else 'black')
plt.colorbar(im, fraction=0.04); ax.set_title('Supp. Fig. S1. Dispersion-feature correlations (confirmation)')
savefig('FigS1_dispersion_correlations'); plt.show()

### 6.9 Figure 4 — Kaplan–Meier and forest plot
**(A)** Overall survival by CD8_gini median split with numbers at risk and log-rank p. **(B)** Hazard ratios per SD (95% CI, log scale) for every Study 1 model; the primary is in red. All intervals crossing 1 = null.

In [ ]:
from lifelines.plotting import add_at_risk_counts
fig = plt.figure(figsize=(7.4, 3.8))
ax = fig.add_axes([0.10, 0.30, 0.38, 0.62]); kms = []
for m, lab, col in [(hi_grp, f'CD8 Gini above median (n={hi_grp.sum()})', BLUE),
                    (~hi_grp, f'CD8 Gini at/below median (n={(~hi_grp).sum()})', ORANGE)]:
    k = KaplanMeierFitter(label=lab).fit(CONF.time_days[m] / 30.44, CONF.event[m])
    k.plot_survival_function(ax=ax, ci_show=False, color=col, show_censors=True, censor_styles={'ms': 3}); kms.append(k)
ax.set_xlim(0, 120); ax.set_ylim(0, 1.02); ax.set_xlabel('Months'); ax.set_ylabel('Overall survival')
ax.text(60, 0.9, f'Log-rank p = {lr.p_value:.2f}', fontsize=8); ax.legend(frameon=False, fontsize=7, loc='lower left')
ax.set_title('A', loc='left', fontweight='bold')
add_at_risk_counts(*kms, ax=ax, rows_to_show=['At risk'], labels=['Above', 'At/below'])

ax2 = fig.add_axes([0.68, 0.18, 0.30, 0.74]); t = table2.iloc[::-1].reset_index(drop=True)
for i, r in t.iterrows():
    ax2.plot([r.lo, r.hi], [i, i], color='k', lw=1)
    ax2.plot(r.HR, i, 's', color=RED if r.analysis == 'Primary' else BLUE, ms=5)
ax2.set_yticks(range(len(t))); ax2.set_yticklabels([f'{a}: {f}' if a != f else f for a, f in zip(t.analysis, t.feature)], fontsize=7)
ax2.axvline(1, color='k', ls=':', lw=0.8); ax2.set_xscale('log')
ax2.set_xticks([0.8, 1, 1.25]); ax2.set_xticklabels(['0.8', '1.0', '1.25']); ax2.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
ax2.set_xlabel('HR per SD (95% CI)'); ax2.set_title('B', loc='left', fontweight='bold')
savefig('Figure4_survival'); plt.show()

### 6.10 Save Table 2
Writes all Study 1 results (gate, primary, C-index, secondaries, KM) to `outputs/Table2_study1.csv`.

In [ ]:
t2 = table2.copy()
t2.loc[len(t2)] = dict(analysis='Validation gate (co-primary)', feature='CD8_gini vs saltz_gini', HR=np.nan,
                       lo=gate['lo'], hi=gate['hi'], p=gate['p'], n=gate['n'], events=np.nan)
t2['note'] = ''
t2.loc[t2.analysis == 'Primary', 'note'] = f'OOF C-index (5-fold, seed 42) = {c_kf:.3f}'
t2.loc[t2.analysis.str.startswith('Validation'), 'note'] = f"rho = {gate['rho']:.3f}; threshold {GATE_RHO}"
t2.to_csv(OUT_DIR / 'Table2_study1.csv', index=False); t2.round(3)

## Part 7 — Study 2: molecular validation (plan of 6 Sep 2026; retrospectively registered osf.io/jdqas)

Follows the written plan, **not** the first analysis run: H1 uses the **Ayers 18-gene IFN-γ signature** (the original run used the Thorsson score as primary — that is now an additional analysis). Uses slide-level **means** of all 23 channels across the full 456-patient cohort; no exploratory/confirmation split (plan §3).

### 7.1 Load molecular data
Three public sources, saved in `molecular/`:

- `thorsson_immune_landscape.xlsx` — Thorsson et al. 2018 Table S1 (IFN-γ Response, CIBERSORT T Cells CD8).
- `tcga_luad_tpm.tsv` — UCSC Xena TCGA-LUAD RNA-seq, genes × samples.
- `tcga_luad_rppa.tsv` — TCPA RPPA, samples × proteins (PD-L1 column).

Sample barcodes are cut to the 12-character patient ID; for RNA and RPPA only primary-tumour samples (`-01`) are kept.

In [ ]:
def find_col(df, *keys):
    for c in df.columns:
        if all(k.lower() in str(c).lower() for k in keys): return c
    raise KeyError(f'no column with {keys}')

th = pd.read_excel(MOL_DIR / 'thorsson_immune_landscape.xlsx')
th = th[th['TCGA Study'] == 'LUAD']
thorsson = pd.DataFrame({'patient_barcode': th['TCGA Participant Barcode'].str[:12],
                         'ifng_thorsson': pd.to_numeric(th['IFN-gamma Response'], errors='coerce'),
                         'cibersort_cd8': pd.to_numeric(th['T Cells CD8'], errors='coerce')})

tpm = pd.read_csv(MOL_DIR / 'tcga_luad_tpm.tsv', sep='\t', index_col=0)
tpm = tpm.loc[:, [c for c in tpm.columns if str(c)[13:15] == '01']]
tpm.columns = [c[:12] for c in tpm.columns]; tpm = tpm.T.groupby(level=0).mean().T

rppa = pd.read_csv(MOL_DIR / 'tcga_luad_rppa.tsv', sep='\t')
sid = find_col(rppa, 'sample') if any('sample' in c.lower() for c in rppa.columns) else rppa.columns[0]
rppa = rppa[rppa[sid].astype(str).str[13:15] == '01']
pdl1_col = [c for c in rppa.columns if c.replace('-', '').replace('_', '').upper().startswith('PDL1')][0]
rppa_pdl1 = pd.DataFrame({'patient_barcode': rppa[sid].str[:12], 'rppa_pdl1': rppa[pdl1_col]}).groupby('patient_barcode').mean()
print('Thorsson LUAD rows:', len(thorsson), '| RNA patients:', tpm.shape[1], '| RPPA patients:', len(rppa_pdl1), f'(column {pdl1_col})')

### 7.2 Ayers IFN-γ signature score (H1 primary measure) ⚠️ check
Plan §4: mean of z-scored log2(TPM+1) across the 18 genes of Ayers et al. 2017. The list below is the published 18-gene T-cell-inflamed expression profile. ⚠️ Your plan text named some genes from Ayers' smaller IFN-γ panels (e.g. IFNG, CXCL10, GZMA); confirm which list your original run used and edit `AYERS_GENES` to match — this was registered as "gene list as implemented in the analysis code".

Xena TPM files are often already log2(TPM+0.001); the cell detects this (max value < 30) and back-transforms before applying log2(TPM+1).

In [ ]:
AYERS_GENES = ['CCL5', 'CD27', 'CD274', 'CD276', 'CD8A', 'CMKLR1', 'CXCL9', 'CXCR6', 'HLA-DQA1',
               'HLA-DRB1', 'HLA-E', 'IDO1', 'LAG3', 'NKG7', 'PDCD1LG2', 'PSMB10', 'STAT1', 'TIGIT']
expr = tpm.reindex(AYERS_GENES)
missing = expr.index[expr.isna().all(axis=1)].tolist()
if missing: print('⚠️ genes not found:', missing)
expr = expr.dropna(how='all')
if np.nanmax(expr.values) < 30:            # already log2(TPM + 0.001)
    expr = np.log2((2 ** expr - 0.001).clip(lower=0) + 1)
else:
    expr = np.log2(expr + 1)
z = expr.sub(expr.mean(axis=1), axis=0).div(expr.std(axis=1), axis=0)
ayers = z.mean(axis=0).rename('ifng_ayers').to_frame()
print('Ayers score for', len(ayers), 'patients using', len(expr), 'genes')

### 7.3 Assemble the Study 2 dataset
Joins the 23-channel slide means for all 456 patients to the four molecular measures, and builds the **global 23-channel mean** used in the forensics.

In [ ]:
mean_cols = [f'{c}_mean' for c in CHANNELS if f'{c}_mean' in data]
assert len(mean_cols) == 23, f'Study 2 needs all 23 channel means (found {len(mean_cols)}) — run Part 4.1 with per-tile files.'
S2 = (data[['patient_barcode'] + mean_cols]
      .merge(ayers, left_on='patient_barcode', right_index=True, how='left')
      .merge(thorsson, on='patient_barcode', how='left')
      .merge(rppa_pdl1, left_on='patient_barcode', right_index=True, how='left'))
S2['global_mean'] = S2[mean_cols].mean(axis=1)   # plan §5.4: global mean across all 23 channels
print(S2[['ifng_ayers', 'ifng_thorsson', 'cibersort_cd8', 'rppa_pdl1']].notna().sum())

### 7.4 Hypotheses H1–H3 (plan §5.1–5.2) and the additional Thorsson analysis
Spearman ρ, bootstrap 95% CI and p, judged against the plan's thresholds (ρ **and** p). Expected from the original data: H1 ρ ≈ 0.138 (n = 451), H2 ρ ≈ 0.072, H3 ρ ≈ 0.193 — all not met. The Thorsson IFN-γ comparison was not in the plan and is labelled as additional.

In [ ]:
tests = [('H1 (primary)', 'CD8_mean', 'ifng_ayers', H1_RHO, H1_P),
         ('H2', 'CD8_mean', 'cibersort_cd8', H2_RHO, H2_P),
         ('H3', 'PD-L1_mean', 'rppa_pdl1', H3_RHO, H3_P),
         ('Additional (not in plan)', 'CD8_mean', 'ifng_thorsson', np.nan, np.nan)]
out = []
for name, x, y, rt, pt in tests:
    r = spearman_boot(S2[x], S2[y])
    met = '—' if np.isnan(rt) else ('met' if (r['rho'] >= rt and r['p'] < pt) else 'not met')
    out.append({'hypothesis': name, 'virtual': x, 'ground_truth': y, **r, 'threshold': rt, 'outcome': met})
table3 = pd.DataFrame(out); table3.to_csv(OUT_DIR / 'Table3_study2.csv', index=False)
table3.round(3)

### 7.5 Scatter plots for H1–H3 (Supplementary Figure S2)
One panel per hypothesis: virtual marker (x) against molecular ground truth (y), with ρ, n and the threshold in the title. Visual check that no outlier drives the correlations.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
for ax, (_, r) in zip(axes, table3.iloc[:3].iterrows()):
    d = S2[[r.virtual, r.ground_truth]].dropna()
    ax.scatter(d.iloc[:, 0], d.iloc[:, 1], s=6, alpha=0.5, color=BLUE)
    ax.set_xlabel(f'Virtual {r.virtual}'); ax.set_ylabel(r.ground_truth)
    ax.set_title(f"{r.hypothesis}: ρ={r.rho:.3f} (n={r.n})\nthreshold ρ≥{r.threshold}", fontsize=8)
plt.tight_layout(); savefig('FigS2_study2_scatter'); plt.show()

### 7.6 Model-recovery ordering (plan §5.3) — Figure 3
Correlates **all 23 channels** with the IFN-γ score and ranks them. Per the plan the reference is the **Ayers** signature (the original run used Thorsson — both are computed so the deviation can be shown). Plan expectation: immune channels (CD3, CD8, CD4, CD20, PD-L1, PD-1, CD68) rank above structural ones (CK, actin, transgelin). A dashed line marks the global-mean correlation.

In [ ]:
IMMUNE = {'CD3', 'CD8', 'CD4', 'CD20', 'PD-L1', 'PD-1', 'CD68', 'CD14', 'CD11c', 'CD16', 'T-bet', 'CD138'}
def channel_ranking(ref):
    rows = []
    for c in mean_cols:
        rho, p = stats.spearmanr(S2[c], S2[ref], nan_policy='omit')
        name = c[:-5]; rows.append({'channel': name, 'category': 'immune' if name in IMMUNE else 'structural', 'rho': rho, 'p': p})
    return pd.DataFrame(rows).sort_values('rho', ascending=False).reset_index(drop=True)

rank_ayers, rank_thor = channel_ranking('ifng_ayers'), channel_ranking('ifng_thorsson')
rank_ayers.to_csv(OUT_DIR / 'channel_ranking_ayers.csv', index=False); rank_thor.to_csv(OUT_DIR / 'channel_ranking_thorsson.csv', index=False)
glob_a = stats.spearmanr(S2.global_mean, S2.ifng_ayers, nan_policy='omit')[0]

d = rank_ayers.iloc[::-1]
fig, ax = plt.subplots(figsize=(4.6, 5.6))
ax.barh(d.channel, d.rho, color=[BLUE if c == 'immune' else GREY for c in d.category], height=0.7)
ax.axvline(0, color='k', lw=0.6); ax.axvline(glob_a, color=RED, ls='--', lw=0.9)
ax.text(glob_a, len(d) - 1, f' global mean\n ρ={glob_a:.3f}', color=RED, fontsize=7, va='top')
for t in ax.get_yticklabels():
    if t.get_text() in ('CD8', 'CK'): t.set_fontweight('bold')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=BLUE, label='Immune'), Patch(color=GREY, label='Structural / other')], frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.09), ncol=2)
ax.set_xlabel('Spearman ρ with Ayers IFN-γ signature'); ax.set_title('Figure 3. Channel ranking (plan §5.3)')
savefig('Figure3_channel_ranking'); plt.show()
print('Mean rho — immune:', rank_ayers.query("category=='immune'").rho.mean().round(3),
      '| structural:', rank_ayers.query("category=='structural'").rho.mean().round(3))

### 7.7 Artefact forensics (plan §5.4)
Compares the IFN-γ correlation of virtual CD8 with that of the **global 23-channel mean** and of **CK** (negative control), and computes the partial correlation of CD8 with IFN-γ given the global mean. If the global mean or CK correlates about as strongly as CD8, H1 evidence is downgraded (plan §7b). Done for both IFN-γ measures.

In [ ]:
fx = []
for ref in ['ifng_ayers', 'ifng_thorsson']:
    fx.append({'reference': ref,
               'CD8': stats.spearmanr(S2.CD8_mean, S2[ref], nan_policy='omit')[0],
               'global_mean': stats.spearmanr(S2.global_mean, S2[ref], nan_policy='omit')[0],
               'CK': stats.spearmanr(S2.CK_mean, S2[ref], nan_policy='omit')[0],
               'partial_CD8_given_global': partial_spearman(S2.CD8_mean, S2[ref], S2.global_mean)})
forensics = pd.DataFrame(fx); forensics.to_csv(OUT_DIR / 'study2_forensics.csv', index=False); forensics.round(3)

### 7.8 Interpretation per the plan's matrix (§7)
Applies the pre-written rules automatically: (a) H1 passes and forensics clean; (b) H1 passes but CK or global mean correlates comparably; (c) H1 fails — complementary null.

In [ ]:
h1 = table3.iloc[0]; fa = forensics.iloc[0]
if h1.outcome == 'met':
    verdict = '(b) H1 passes but non-specific' if max(abs(fa.global_mean), abs(fa.CK)) >= 0.8 * fa.CD8 else '(a) H1 passes, forensics clean'
else:
    verdict = '(c) H1 fails: virtual CD8 does not robustly recover the IFN-γ program'
print('Interpretation:', verdict)

## Part 8 — Table 1 and Figure 1

### 8.1 Table 1: cohort characteristics
Age (median, IQR), sex, AJCC stage, deaths, median follow-up (reverse Kaplan–Meier) and median overall survival for the exploratory, confirmation and pooled sets.

In [ ]:
def describe(d):
    a = d.age; fu = KaplanMeierFitter().fit(d.time_days / 30.44, 1 - d.event).median_survival_time_
    os_ = KaplanMeierFitter().fit(d.time_days / 30.44, d.event).median_survival_time_
    n = len(d); pct = lambda k: f'{k} ({100*k/n:.1f})'
    row = {'N': n, 'Age, median (IQR)': f'{a.median():.0f} ({a.quantile(.25):.0f}-{a.quantile(.75):.0f})',
           'Female, n (%)': pct((d.sex.astype(str).str.lower() == 'female').sum())}
    for s in ['I', 'II', 'III', 'IV']: row[f'Stage {s}, n (%)'] = pct((d.stage4 == s).sum())
    row['Stage missing, n'] = int(d.stage4.isna().sum())
    row['Deaths, n (%)'] = pct(int(d.event.sum()))
    row['Median follow-up, months'] = round(fu, 1); row['Median OS, months'] = round(os_, 1)
    return pd.Series(row)

table1 = pd.DataFrame({'Exploratory': describe(EXPL), 'Confirmation': describe(CONF), 'Pooled': describe(data)})
table1.to_csv(OUT_DIR / 'Table1_cohort.csv'); table1

### 8.2 Figure 1: study design and cohort flow
Boxes-and-arrows diagram of the two studies, with counts filled in from this run. (TLCR asks for flow diagrams in an editable format — rebuild this one in PowerPoint for submission and use the PNG as a guide.)

In [ ]:
n_ex, n_cf = len(EXPL), len(CONF); d_ex, d_cf = int(EXPL.event.sum()), int(CONF.event.sum())
fig, ax = plt.subplots(figsize=(7.2, 4.6)); ax.axis('off'); ax.set_xlim(0, 10); ax.set_ylim(0, 10)
def box(x, y, w, h, t, fc='#eef3f8'):
    ax.add_patch(plt.Rectangle((x, y), w, h, fc=fc, ec='#333', lw=0.8)); ax.text(x + w/2, y + h/2, t, ha='center', va='center', fontsize=7.6)
def arr(x1, y1, x2, y2): ax.annotate('', (x2, y2), (x1, y1), arrowprops=dict(arrowstyle='->', lw=0.8))
box(2.6, 8.6, 4.8, 1.1, f'TCGA-LUAD patients with diagnostic H&E\nand overall-survival data (n={n_ex + n_cf + 9})')
box(7.4, 7.3, 2.5, 0.9, 'Excluded: anomalous\nfile size (n=9)', fc='#f6f6f6'); arr(5, 8.6, 5, 7.6); arr(5, 7.75, 7.4, 7.75)
box(2.6, 6.6, 4.8, 1.0, f'GigaTIME-Flash, mpp-aware tiling\n(128 µm → 256 px); n={n_ex + n_cf}')
arr(4.2, 6.6, 2.3, 5.6); arr(5.8, 6.6, 7.7, 5.6)
box(0.6, 4.6, 3.4, 1.0, f'Exploratory set\nn={n_ex}, {d_ex} deaths'); box(6, 4.6, 3.4, 1.0, f'Confirmation set\nn={n_cf}, {d_cf} deaths')
arr(2.3, 4.6, 2.3, 3.6); box(0.1, 2.5, 4.4, 1.2, 'Pipeline development and\nfeature choice → pre-registration\n(osf.io/kduvj, 6 Sep 2026)')
arr(4.5, 3.1, 6, 3.1); arr(7.7, 4.6, 7.7, 3.6)
box(6, 2.4, 3.6, 1.2, 'Study 1: Saltz gate (ρ≥0.30)\n+ Cox OS on CD8_gini\n+ prespecified secondaries')
box(3.2, 0.3, 6.4, 1.2, f'Study 2 (plan 6 Sep 2026; osf.io/jdqas), n={n_ex + n_cf}:\nvirtual means vs Ayers IFN-γ (H1), CIBERSORT CD8 (H2), RPPA PD-L1 (H3);\nchannel ordering; shared-factor test')
arr(7.7, 2.4, 7.7, 1.5)
savefig('Figure1_study_design'); plt.show()

## Part 9 — Export and summary

### 9.1 Save the analysis dataset and a results summary
Writes the merged patient-level dataset and a one-page JSON of the headline numbers (for pasting into the manuscript), then lists every file in `outputs/`.

In [ ]:
data.to_csv(OUT_DIR / 'analysis_dataset.csv', index=False)
summary = {
    'study1': {'gate_rho': round(gate['rho'], 3), 'gate_pass': bool(gate_pass),
               'primary_HR': round(prim['HR'], 3), 'primary_CI': [round(prim['lo'], 2), round(prim['hi'], 2)],
               'primary_p': round(prim['p'], 3), 'oof_C_kfold': round(c_kf, 3), 'n': prim['n'], 'events': prim['events']},
    'study2': table3[['hypothesis', 'rho', 'lo', 'hi', 'p', 'n', 'outcome']].round(3).to_dict('records'),
    'study2_interpretation': verdict,
}
(OUT_DIR / 'results_summary.json').write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps(summary, indent=2, default=str))
print('\nOutputs:'); [print(' ', p.name) for p in sorted(OUT_DIR.iterdir())]

### 9.2 Session info
Records package versions for the Methods section ("Analyses used Python …").

In [ ]:
import sys, sklearn, lifelines, scipy
print('Python', sys.version.split()[0])
for m in [np, pd, scipy, lifelines, sklearn, matplotlib]:
    print(f'{m.__name__:12s}', m.__version__)